In [1]:
import pandas as pd
from darts.timeseries import TimeSeries
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from darts.dataprocessing.transformers import Scaler
from darts.models import TFTModel
from darts.dataprocessing.transformers import StaticCovariatesTransformer
import numpy as np
import torch
import matplotlib.pyplot as plt

c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


### Loss Logger 

In [7]:
from pytorch_lightning.callbacks import Callback

class LossLogger(Callback):
    """
    A PyTorch Lightning callback to record training and validation losses 
    at the end of every epoch for custom plotting or analysis.
    """
    def __init__(self):
        super().__init__()
        self.train_losses = []
        self.val_losses = []

    def on_train_epoch_end(self, trainer, pl_module):
        # Retrieve train_loss from callback_metrics
        train_loss = trainer.callback_metrics.get("train_loss")
        
        if train_loss is not None:
            # detach() ensures we don't keep the computation graph in memory
            # cpu() ensures it works regardless of whether you're on GPU or CPU
            self.train_losses.append(float(train_loss.detach().cpu()))

    def on_validation_epoch_end(self, trainer, pl_module):
        # Retrieve val_loss from callback_metrics
        val_loss = trainer.callback_metrics.get("val_loss")
        
        if val_loss is not None:
            self.val_losses.append(float(val_loss.detach().cpu()))

### Add encoders

In [8]:
# Function to encode the year as a normalized value
def encode_year(idx):
  return (idx.year - 2000) / 50

def encode_days_in_month(index):
  return index.days_in_month.to_numpy().reshape(-1,1)

# Set up the add_encoders dictionary to specify how different time-related encoders and transformers should be applied
add_encoders = {
    'cyclic': {'past': ['month'], 'future': ['month']},
    'position': {'past': ['relative'], 'future': ['relative']},
    'custom': {
        'past': [encode_year, encode_days_in_month],
        'future': [encode_year, encode_days_in_month]
    },
    'transformer': Scaler()
}

### Read the data

In [10]:
pandas_df = pd.read_csv(r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series\all_series_data.csv",index_col = ['MONTH_OF_SALE'],parse_dates=True)

### Separating the type of variables

In [11]:
#Extracting type of columns according to the datatypes
# 1. Targets/Metrics (The numbers we want to predict)
target_cols = pandas_df.select_dtypes(include=['number']).columns.tolist()
target_cols.append('PARENT_DEALER_CODE_MODEL_FAMILY')

# 2. Time Dimension
time_cols = pandas_df.select_dtypes(include=['datetime', 'datetime64']).columns.tolist()

# 3. Static/Categorical Covariates (The identifiers)
# We exclude numbers and dates to find the "ID" strings
static_cols = pandas_df.select_dtypes(exclude=['number', 'datetime', 'datetime64']).columns.tolist()

print(f"Targets: {target_cols}")
print(f"Time Column: {time_cols}")
print(f"Static Identifiers: {static_cols}")

Targets: ['DEALER_CODE', 'NUM_FESTIVE_DAYS_MONTH', 'FESTIVE_PHASE_I', 'FESTIVE_PHASE_II', 'FESTIVE_PHASE_III', 'TOTAL_DAYS_FESTIVE_PHASE_I', 'TOTAL_DAYS_FESTIVE_PHASE_II', 'TOTAL_DAYS_FESTIVE_PHASE_III', 'TOTAL_DAYS_PITRU_PAKSH', 'PROP_FESTIVE_PHASE_I', 'PROP_EVENT_FESTIVE_PHASE_I', 'PROP_FESTIVE_PHASE_II', 'PROP_EVENT_FESTIVE_PHASE_II', 'PROP_FESTIVE_PHASE_III', 'PROP_EVENT_FESTIVE_PHASE_III', 'PROP_PITRU_PAKSH', 'PROP_EVENT_PITRU_PAKSH', 'NET_SALES', 'AKSHAYA_TRITIYA_DAYS', 'BHAI_DOOJ_DAYS', 'BUDDHA_PURNIMA_DAYS', 'CHHATH_PUJA_DAYS', 'DHANTERAS_DAYS', 'DIWALI_DAYS', 'DUSSEHRA_(VIJAYADASHAMI)_DAYS', 'EID_UL_FITR_DAYS', 'GANESH_CHATURTHI_DAYS', 'GANGA_DUSSEHRA_DAYS', 'GOVARDHAN_POOJA_DAYS', 'GURU_PURNIMA_DAYS', 'HANUMAN_JAYANTI_DAYS', 'HARTALIK_TEEJ_DAYS', 'HOLI_DAYS', 'HOLIKA_DAHAN_DAYS', 'JAGANNATH_RATHYATRA_DAYS', 'JANMASHTAMI_DAYS', 'KARWA_CHAUTH_DAYS', 'LOHRI_DAYS', 'MAHA_SHIVARATRI_DAYS', 'MAKAR_SANKRANTI_PONGAL_DAYS', 'NAG_PANCHAMI_DAYS', 'NAVRATRI_DAYS', 'NEW_YEAR_DAYS', 'ONAM_

In [12]:
#Separating the covariates
target_col = ["NET_SALES"]

#future covariates
future_covariates = [i for i in target_cols if i!='NET_SALES']

#actual_static_cols
actual_static_cols = [i for i in static_cols if i!='PARENT_DEALER_CODE_MODEL_FAMILY']

In [13]:
#since variables like MODEL_FAMILY,BRAKE_VARIANT,IGNITION_TYPE,WHEEL_TYPE,BIKE_COLOUR are mostly same for all the top 10 series, will be removing them from the static covariates'
static_covariates = [i for i in actual_static_cols if i not in ['MODEL_FAMILY','BRAKE_VARIANT','IGNITION_TYPE','WHEEL_TYPE','BIKE_COLOUR','DEALER_CODE']]
static_covariates

['DEALER_CITY', 'X_CITY_CATEGORY']

### Preparing data for Darts

In [14]:
#Step 1 - Sorting the dataframe by date
pandas_df.reset_index().sort_values(by=["PARENT_DEALER_CODE_MODEL_FAMILY","MONTH_OF_SALE"]).set_index("MONTH_OF_SALE",inplace=True)

In [15]:
#Step 2 - Separating the static covariates and NET_SALES column
pandas_df_with_target_and_static_covariates = pandas_df.loc[:,['PARENT_DEALER_CODE_MODEL_FAMILY','NET_SALES']+static_covariates]
pandas_df_with_target_and_static_covariates.head()

,PARENT_DEALER_CODE_MODEL_FAMILY,NET_SALES,DEALER_CITY,X_CITY_CATEGORY
MONTH_OF_SALE,,,,
2023-04-01,11211_XPULSE_DISC<>SELF<>SPOKE<>SPORTS RED,1.0,HAZARIBAG,UNSPECIFIED
2023-04-01,11682_PLEASURE+_DRUM<>SELF<>CAST<>BLUE,1.0,SAMBHAL,URBAN
2023-04-01,11681_SPLENDOR+_DRUM<>SELF<>CAST<>PRIME RED,2.0,KOLKATA,URBAN
2023-04-01,11073_GLAMOUR_DRUM<>SELF<>CAST<>BLUE BLACK,0.0,KURNOOL,RURAL
2023-04-01,11641_DESTINI_DRUM<>SELF<>CAST<>BROWN,0.0,CHANDIKHOLE,RURAL


In [16]:
#Step 3 - Separating the future covariates
pandas_df_with_future_covariates = pandas_df.loc[:,future_covariates]
pandas_df_with_future_covariates.head()

,DEALER_CODE,NUM_FESTIVE_DAYS_MONTH,FESTIVE_PHASE_I,FESTIVE_PHASE_II,FESTIVE_PHASE_III,TOTAL_DAYS_FESTIVE_PHASE_I,TOTAL_DAYS_FESTIVE_PHASE_II,TOTAL_DAYS_FESTIVE_PHASE_III,TOTAL_DAYS_PITRU_PAKSH,PROP_FESTIVE_PHASE_I,...,NAVRATRI_DAYS,NEW_YEAR_DAYS,ONAM_DAYS,PITRAPAKSHA_DAYS,RAKSHA_BANDHAN_DAYS,REPUBLIC_DAY_DAYS,VASANT_PANCHAMI_DAYS,VISHWAKARMA_PUJA_DAYS,MARRIAGE_DAYS,PARENT_DEALER_CODE_MODEL_FAMILY
MONTH_OF_SALE,,,,,,,,,,,,,,,,,,,,,
2023-04-01,11211,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11211_XPULSE_DISC<>SELF<>SPOKE<>SPORTS RED
2023-04-01,11682,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11682_PLEASURE+_DRUM<>SELF<>CAST<>BLUE
2023-04-01,11681,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11681_SPLENDOR+_DRUM<>SELF<>CAST<>PRIME RED
2023-04-01,11073,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11073_GLAMOUR_DRUM<>SELF<>CAST<>BLUE BLACK
2023-04-01,11641,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11641_DESTINI_DRUM<>SELF<>CAST<>BROWN


In [17]:
#Step 4 - Creating the darts timeseries object for target and static covariates
darts_df_with_static_covariates = TimeSeries.from_group_dataframe(df=pandas_df_with_target_and_static_covariates,
                                                                  group_cols=["PARENT_DEALER_CODE_MODEL_FAMILY"],
                                                                  static_cols=static_covariates,value_cols=["NET_SALES"],freq='MS')


In [18]:
#Step 5 - Creating the darts timeseries object with future covariates

#Removing PARENT_DEALER_CODE_MODEL_FAMILY from future_covariates
try:
    future_covariates.remove('PARENT_DEALER_CODE_MODEL_FAMILY')
except:
    pass

darts_df_with_future_covariates = TimeSeries.from_group_dataframe(df = pandas_df_with_future_covariates,
                                    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
                                    freq = 'MS',
                                    value_cols = future_covariates
                                    )

### Train/Test split

In [19]:
#Step 6 - Creating train, test, and validation split
#Train set - Apr'23 to Dec'25 
#Val set - Jan'26 to Mar'26 


train_list = []
val_list = []

for ts in darts_df_with_static_covariates:
    train = ts.slice(pd.Timestamp('2023-04-01'), pd.Timestamp('2025-12-01'))
    val = ts.slice(pd.Timestamp('2025-01-01'), pd.Timestamp('2026-03-01'))
    
    train_list.append(train)
    val_list.append(val)

train_future_covariates_list = []
validation_future_covariates_list = []

for ts in darts_df_with_future_covariates:
    train = ts.slice(pd.Timestamp('2023-04-01'), pd.Timestamp('2025-12-01'))
    val = ts.slice(pd.Timestamp('2025-01-01'), pd.Timestamp('2026-03-01'))
    train_future_covariates_list.append(train)
    validation_future_covariates_list.append(val)

### Scaling

In [20]:

target_scaler = Scaler(n_jobs=-1)
future_covariates_scaler = Scaler(n_jobs=-1)

transformer = StaticCovariatesTransformer(n_jobs=-1)

#Scale the target training data
scaled_target_series = target_scaler.fit_transform(train_list)

scaled_target_series_with_static_covariates_training = transformer.fit_transform(scaled_target_series)



# #Scale the static covariates in training data
# scaled_static_covariates_training = transformer.fit_transform(train_list)

# #Scale the future covariates in training data
# # scaled_future_covariates = future_covariates_scaler.fit_transform(darts_df_with_future_covariates)

scaled_future_covariates_training = future_covariates_scaler.fit_transform(train_future_covariates_list)
scaled_future_covariates_validation = future_covariates_scaler.transform(validation_future_covariates_list)


# #Scale the target validation data
scaled_target_series_validation = target_scaler.transform(val_list)
scaled_target_series_with_static_covariates_validation = transformer.transform(scaled_target_series_validation)

# #Scale the static covariates in validation data
# scaled_static_covariates_validation = transformer.transform(val_list)



### Save the scaler objects

In [21]:
import joblib 

joblib.dump(target_scaler,r"scaler_objects/target_scaler.pkl")
joblib.dump(future_covariates_scaler,r"scaler_objects/future_covariates_scaler.pkl")
joblib.dump(transformer,r"scaler_objects/static_covariate_transformer.pkl")



['scaler_objects/static_covariate_transformer.pkl']

### Training the model

In [25]:
from datetime import datetime
from pytorch_lightning.callbacks import ModelCheckpoint

current_date = datetime.now().strftime("%Y-%m-%d")

MODEL_NAME = f"tft_net_sales_whole_dataset_{current_date}"

WORK_DIR = r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series"

# 2. Add it to the filename string
custom_checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    dirpath=rf"{WORK_DIR}\{MODEL_NAME}\checkpoints",
    filename=f"tft-best-model-{current_date}-{{epoch:02d}}-{{val_loss:.4f}}",
    save_top_k=1,
    mode="min"
)

### Defining the model architecture

In [26]:
loss_logger = LossLogger()
model = TFTModel(input_chunk_length=12,output_chunk_length=3,
                 batch_size=512,dropout=0.0,likelihood=None,loss_fn=torch.nn.MSELoss(),
                 n_epochs=100,random_state=42,add_encoders=add_encoders,model_name=MODEL_NAME,
                 work_dir = WORK_DIR,save_checkpoints = True,force_reset=True,
                 pl_trainer_kwargs={'callbacks': [loss_logger,custom_checkpoint_callback],
                                    'enable_checkpointing':True})

model.fit(series=scaled_target_series_with_static_covariates_training,
          future_covariates = scaled_future_covariates_training,
          val_series = scaled_target_series_with_static_covariates_validation,
          val_future_covariates=scaled_future_covariates_validation,verbose=True)

trainer = model.trainer
optimizer = trainer.optimizers[0]
current_lr = optimizer.param_groups[0]["lr"]
total_epochs = model.trainer.max_epochs

def title_of_the_plot():
    total_epochs = len(loss_logger.losses)
    
    if total_epochs > 0:
        last_loss = loss_logger.losses[-1]
        return f"Overfitting Debug - lr: {current_lr}, epochs: {total_epochs}, last_loss: {last_loss:.2f}"
    return "Overfitting Debug - No Data"

plt.figure(figsize=(10, 6))
plt.plot(loss_logger.losses)
plt.xlabel('Epoch')
plt.ylabel('Train Loss')
plt.title(title_of_the_plot())
plt.grid(True, alpha=0.3) 
plt.show()

RuntimeError: Found more than one stateful callback of type `ModelCheckpoint`. In the current configuration, this callback does not support being saved alongside other instances of the same type. Please consult the documentation of `ModelCheckpoint` regarding valid settings for the callback state to be checkpointable. HINT: The `callback.state_key` must be unique among all callbacks in the Trainer.

### Loading the best checkpoint

In [22]:
model = TFTModel(
    input_chunk_length=12,
    output_chunk_length=3,
    batch_size=512,
    dropout=0.0,
    likelihood=None,
    loss_fn=torch.nn.MSELoss(),
    n_epochs=100,
    random_state=42,
    add_encoders=add_encoders
)

In [23]:
checkpoint_path = r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series\checkpoints\tft-best-model-2026-05-01-epoch=00-val_loss=0.1076.ckpt"

In [24]:
model.load_from_checkpoint(checkpoint_path)

ValueError: Could not find base model save file `_model.pth.tar` in C:\Users\G0004878\Desktop\TFT_Data\Multi_series\checkpoints\tft-best-model-2026-05-01-epoch=00-val_loss=0.1076.ckpt.


ValueError: Could not find base model save file `_model.pth.tar` in C:\Users\G0004878\Desktop\TFT_Data\Multi_series\checkpoints\tft-best-model-2026-05-01-epoch=00-val_loss=0.1076.ckpt.